# Arabic Sentiment Foundation Model -- Documentation & Reproduction Notebook

**This notebook loads already-saved artifacts and does NOT retrain anything.** All training happened in `scripts/arabic_foundation_*.py`, run once, with results frozen under `artifacts/experiments/arabic_foundation/` and `reports/generated/arabic_foundation/`.

## Objective

Build, validate, freeze, and document the strongest scientifically defensible **Arabic sentiment FOUNDATION model** from currently available non-Jumia data. This is explicitly **NOT** a claim of Egyptian e-commerce production readiness -- it is an Arabic sentiment foundation model, intended for later Egyptian e-commerce domain adaptation.

**Primary task**: 3-class sentiment, 0=Negative, 1=Neutral/Mixed, 2=Positive, trained on LABR (Large-scale Arabic Book Reviews).

## Jumia exclusion rationale

Per explicit project scope, Jumia data/artifacts/configs are **completely excluded** from every modeling, training, validation, testing, and calibration decision in this task (`JUMIA_WEIGHT_IN_ALL_MODELING_DECISIONS=0`). This notebook never reads any `data/raw/jumia/`, `artifacts/experiments/jumia/`, or Jumia-related config path. The only Jumia-related action anywhere in this task was a final stat/hash confirm-only audit that those paths were untouched -- not reproduced here since it performs no read of Jumia content itself, only `os.stat`/hash checks recorded in the reproducibility manifest.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

REPORTS_DIR = REPO_ROOT / "reports" / "generated" / "arabic_foundation"
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "experiments" / "arabic_foundation"

def load_json(name):
    return json.loads((REPORTS_DIR / name).read_text(encoding="utf-8"))

print("Repo root:", REPO_ROOT)
print("Reports dir exists:", REPORTS_DIR.exists())

Repo root: C:\Users\User2\Desktop\ecommerce_medusa\commerce-pilot-ai
Reports dir exists: True


## Gate 1: Hardware & software audit

In [2]:
hw = load_json("hardware_audit.json")
print("GPU:", hw["gpu"]["name"] if hw["gpu"] else "none")
print("CUDA available:", hw["cuda_available"])
print("torch:", hw["torch_version"], "| transformers:", hw["transformers_version"])
print("bf16 usable:", hw["precision_decision"]["bf16_usable"] if hw["precision_decision"] else None)
print("Genuine GPU compute proof (timed bf16 matmul):", hw["genuine_gpu_compute_proof"])
print("SDPA backend probe (which kernel is actually selected):", hw["sdpa_backend_probe"])
print("torch.compile probe:", hw["torch_compile_probe"])

GPU: NVIDIA RTX 2000 Ada Generation
CUDA available: True
torch: 2.6.0+cu124 | transformers: 5.15.0
bf16 usable: True
Genuine GPU compute proof (timed bf16 matmul): {'ran': True, 'op': '20x bf16 4096x4096 matmul on cuda:0', 'seconds': 0.18384456634521484, 'achieved_tflops': 14.95164705753919, 'peak_memory_allocated_mb': 136.125, 'result_checksum': -428619.0}
SDPA backend probe (which kernel is actually selected): {'flash': 'unavailable: No available kernel. Aborting execution.', 'mem_efficient': 'available', 'math': 'available', 'default_backend_runs_ok': True}
torch.compile probe: {'ran': False, 'error': 'backend=\'inductor\' raised:\nRuntimeError: Cannot find a working triton installation. Either the package is not installed or it is too old. More information on installing Triton can be found at https://github.com/openai/triton\n\nSet TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information\n\n\nYou can suppress this exception and fall back to eager by setting:\n    import 

## Gate 2/3: Dataset selection & LABR full audit

Primary dataset: **LABR** (Large-scale Arabic Book Reviews, Aly & Atiya, ACL 2013). 63,257 raw rows scraped from Goodreads, rating 1-5, GPLv2 license (license recorded but weighted 0 in technical ranking per instruction).

In [3]:
inv = load_json("dataset_inventory.json")
for name, info in inv.items():
    print(f"- {name}: {info.get('role')}" + (f" (n_rows={info['n_rows']})" if 'n_rows' in info else ""))

- labr: PRIMARY training/validation/test dataset (n_rows=63257)
- astd: Cross-domain/dialectal ROBUSTNESS auxiliary only, not training (n_rows=10006)
- arsas: General Arabic sentiment ROBUSTNESS/optional-transfer auxiliary only, not primary training (n_rows=19897)
- mpold: OUT OF SCOPE for sentiment -- offensive-language dataset, labels not used for sentiment training per instruction
- hard_hotel_reviews: UNAVAILABLE, confirmed this session
- arzen: UNAVAILABLE, confirmed this session
- jumia: EXCLUDED FROM ALL MODELING (weight=0), out of scope per hard boundary
- egyptian_tweets_40k: present locally, NOT in priority list for this task, not used
- eesa: not a blocker per instruction, not used


In [4]:
labr_audit = load_json("labr_full_audit.json")
print("n_rows:", labr_audit["n_rows"])
print("rating_counts:", labr_audit["rating_counts"])
print("review_id_is_unique_key:", labr_audit["review_id_is_unique_key"], "<-- data-quality finding, see split builder for the fix (content-derived review_uid)")
print("n_near_duplicate_clusters:", labr_audit["n_near_duplicate_clusters_exact_normalized"])
print("has_timestamp_column:", labr_audit["has_timestamp_column"], "->", labr_audit["timestamp_note"])
print("n_unique_books:", labr_audit["n_unique_books"], "| n_unique_users:", labr_audit["n_unique_users"])

n_rows: 63257
rating_counts: {'1': 2939, '2': 5285, '3': 12201, '4': 19054, '5': 23778}
review_id_is_unique_key: False <-- data-quality finding, see split builder for the fix (content-derived review_uid)
n_near_duplicate_clusters: 2653
has_timestamp_column: False -> reviews.tsv columns are rating/review_id/user_id/book_id/review only -- no timestamp field exists in this release of LABR, confirmed by direct column inspection. Chronological stress split is therefore NOT constructible from this file and is reported as unavailable rather than fabricated (per Gate 5 instruction).
n_unique_books: 2131 | n_unique_users: 16486


## Gate 4: Label contract

Primary 3-class task: LABR rating 1-2 -> **Negative(0)**, 3 -> **Neutral/Mixed(1)**, 4-5 -> **Positive(2)**. The "Neutral/Mixed" class name is deliberately verbose to flag a documented rating-text ambiguity caveat: a 3-star review may reflect genuinely mixed sentiment or a lukewarm-but-directional opinion the rating scale compresses -- a known property of star-rating-derived labels, not a modeling defect.

In [5]:
sys.path.insert(0, str(REPO_ROOT))
from src.nlp.arabic_foundation.normalization import LABEL_NAMES_3CLASS, labr_rating_to_3class
print(LABEL_NAMES_3CLASS)
print({r: labr_rating_to_3class(r) for r in [1,2,3,4,5]})

{0: 'Negative', 1: 'Neutral/Mixed', 2: 'Positive'}
{1: 0, 2: 0, 3: 1, 4: 2, 5: 2}


## Gate 5: Leakage-safe splits

Exact-normalized-text duplicate clusters collapsed to one representative row BEFORE splitting (structurally prevents duplicate content crossing splits). Stratified train/val_natural/test_natural split, val_balanced derived as an equal-per-class subset of val_natural only, and an item-holdout stress split (whole books removed from the training pool). No chronological split -- LABR genuinely has no timestamp column.

In [6]:
sm = load_json("split_manifest.json")
print("Exclusions:", sm["exclusions"])
print()
for name, info in sm["splits"].items():
    print(f"{name}: n={info['n_rows']}, labels={info['label_counts']}")
print()
print("Overlap verification (must show any_overlap_detected=False):")
print(json.dumps(sm["overlap_verification"], indent=2))

Exclusions: {'n_raw': 63257, 'n_dropped_empty_text': 0, 'n_dropped_invalid_rating': 0, 'n_dropped_duplicate_cluster_members': 3204, 'n_rows_after_all_exclusions': 60053}

train: n=47691, labels={'Negative': 6255, 'Neutral/Mixed': 9258, 'Positive': 32178}
val_natural: n=5961, labels={'Negative': 782, 'Neutral/Mixed': 1157, 'Positive': 4022}
val_balanced: n=2346, labels={'Negative': 782, 'Neutral/Mixed': 782, 'Positive': 782}
test_natural: n=5962, labels={'Negative': 782, 'Neutral/Mixed': 1157, 'Positive': 4023}
item_holdout_stress: n=439, labels={'Negative': 53, 'Neutral/Mixed': 93, 'Positive': 293}

Overlap verification (must show any_overlap_detected=False):
{
  "pairwise_review_id_overlap_counts": {
    "train__vs__val_natural": 0,
    "train__vs__test_natural": 0,
    "train__vs__item_holdout_stress": 0,
    "val_natural__vs__test_natural": 0,
    "val_natural__vs__item_holdout_stress": 0,
    "test_natural__vs__item_holdout_stress": 0
  },
  "any_overlap_detected": false,
  "val_ba

## Gate 11: Tokenizer / max-length decision

In [7]:
tok = load_json("token_length_audit.json")
print("Percentiles (full train):", tok["token_length_percentiles_full_train"])
print("Pilot results:", tok["pilot_results"])
print("DECISION:", tok["decision"], "--", tok["decision_reasoning"])

Percentiles (full train): {'50': 41.0, '75': 85.0, '90': 174.0, '95': 266.0, '99': 632.0, 'max': 5836, 'mean': 79.49097313958609}
Pilot results: {'128': {'train_seconds': 19.928013801574707, 'eval_macro_f1': 0.27031311930241775, 'eval_neutral_mixed_f1': 0.0, 'peak_vram_mb': 3690.392578125}, '192': {'train_seconds': 26.51741862297058, 'eval_macro_f1': 0.4800891812043048, 'eval_neutral_mixed_f1': 0.07179487179487179, 'peak_vram_mb': 4489.455078125}}
DECISION: 192 -- macro-F1 gain from 192 vs 128 = +0.2098 (+20.98pp). Decision rule (defined before comparing cost): select 192 only if it improves validation macro-F1 by >=1.0pp; otherwise keep 128 as the cheaper default. Result: selected max_length=192.


## Gate 12: Loss / class-imbalance pilot

In [8]:
loss_pilot = load_json("loss_imbalance_pilot.json")
print("Results:", json.dumps(loss_pilot["results"], indent=2))
print("DECISION:", loss_pilot["decision"])
print(loss_pilot["decision_reasoning"])

Results: {
  "A_standard_ce": {
    "eval_macro_f1": 0.5038347669850961,
    "eval_neutral_mixed_f1": 0.046822742474916385,
    "eval_negative_f1": 0.6103542234332425,
    "eval_positive_f1": 0.8543273350471294
  },
  "B_class_weighted_ce": {
    "eval_macro_f1": 0.6098773142482891,
    "eval_neutral_mixed_f1": 0.39488117001828155,
    "eval_negative_f1": 0.592274678111588,
    "eval_positive_f1": 0.8424760946149975
  }
}
DECISION: B_class_weighted_ce
A (standard CE): macro_f1=0.5038, neutral_mixed_f1=0.0468. B (class-weighted CE): macro_f1=0.6099, neutral_mixed_f1=0.3949. Decision rule: prefer B if it improves BOTH macro-F1 and Neutral/Mixed-F1, or improves Neutral/Mixed-F1 by >2pp without costing >1pp macro-F1; otherwise keep A. Both-fail focal-loss trigger (Neutral/Mixed-F1<0.30 for both): False. Selected: B_class_weighted_ce.


## Gate 10: Classical baseline (word+char TF-IDF + calibrated LinearSVC)

ONE fixed configuration, no hyperparameter search, matching this project's Amazon-pipeline convention adapted for Arabic 3-class.

In [9]:
baseline = load_json("baseline_manifest.json")
print("Model:", baseline["model"])
for split, m in baseline["eval_summary"].items():
    print(f"{split}: macro_f1={m['macro_f1']:.4f} neutral_mixed_f1={m['neutral_mixed_f1']:.4f} acc={m['accuracy']:.4f}")

Model: tfidf_word_char_union + LinearSVC (sigmoid-calibrated via CalibratedClassifierCV, cv=3)
val_natural: macro_f1=0.5148 neutral_mixed_f1=0.1731 acc=0.7331
val_balanced: macro_f1=0.4441 neutral_mixed_f1=0.1844 acc=0.5034
test_natural: macro_f1=0.4964 neutral_mixed_f1=0.1599 acc=0.7259
item_holdout_stress: macro_f1=0.4452 neutral_mixed_f1=0.1321 acc=0.7084


## Gates 13-15: MARBERT primary fine-tune

`UBC-NLP/MARBERT`, end-to-end fine-tune, class-weighted cross-entropy (Gate 12 decision), max_length=192 (Gate 11 decision), lr=2e-5, AdamW, weight_decay=0.01, warmup_steps computed from a 0.06 ratio (transformers 5.15.0 removed the `warmup_ratio` TrainingArguments kwarg -- confirmed by direct `inspect.signature` check), bf16, TF32, best-checkpoint-by-validation-macro-F1, max 3 epochs with an explicit meaningful-improvement-gate (continue past epoch 1 only if BOTH macro-F1 and Neutral/Mixed-F1 improve by >=0.2pp).

In [10]:
mtrain = load_json("marbert_training_manifest.json")
print("Checkpoint:", mtrain["checkpoint"])
print("max_length:", mtrain["max_length"], "| batch_size:", mtrain["batch_size"], "| loss_variant:", mtrain["loss_variant"])
print("epochs actually run:", mtrain["num_epochs_actually_run"], "of", mtrain["num_epochs_configured"], "configured")
print("epoch_history (macro_f1, neutral_mixed_f1) per epoch:", mtrain["epoch_history"])
print("train_seconds:", mtrain["train_seconds"], "| peak_vram_mb:", mtrain["peak_vram_mb"])
print("final_eval_metrics (on val_natural, best checkpoint):", json.dumps(mtrain["final_eval_metrics"], indent=2))

Checkpoint: UBC-NLP/MARBERT
max_length: 192 | batch_size: 128 | loss_variant: B_class_weighted_ce
epochs actually run: 3.0 of 3 configured
epoch_history (macro_f1, neutral_mixed_f1) per epoch: [[0.6001855973426836, 0.37747470303563574], [0.6253879946342299, 0.43840808591282376], [0.6337620595081117, 0.4444444444444444], [0.6337620595081117, 0.4444444444444444]]
train_seconds: 965.7375144958496 | peak_vram_mb: 11986.0625
final_eval_metrics (on val_natural, best checkpoint): {
  "eval_loss": 0.8574575185775757,
  "eval_macro_f1": 0.6337620595081117,
  "eval_weighted_f1": 0.7214441244599685,
  "eval_negative_f1": 0.6398491514770585,
  "eval_neutral_mixed_f1": 0.4444444444444444,
  "eval_positive_f1": 0.8169925826028321,
  "eval_negative_precision": 0.6291718170580964,
  "eval_negative_recall": 0.6508951406649617,
  "eval_neutral_mixed_precision": 0.36839113132461626,
  "eval_neutral_mixed_recall": 0.5600691443388073,
  "eval_positive_precision": 0.89272030651341,
  "eval_positive_recall":

## Gate 16: Calibration (temperature scaling)

In [11]:
calib_path = REPORTS_DIR / "calibration_report.json"
if calib_path.exists():
    calib = json.loads(calib_path.read_text(encoding="utf-8"))
    print(json.dumps(calib, indent=2))
else:
    print("Calibration not yet run.")

{
  "temperature": 1.2616777420043945,
  "val_natural_n": 5961,
  "raw": {
    "brier": 0.4120732995525117,
    "ece": 0.09840540024202715
  },
  "calibrated": {
    "brier": 0.39821351137059485,
    "ece": 0.05468992411220378
  },
  "improved_both_metrics": true,
  "decision": "USE_CALIBRATED",
  "decision_reasoning": "Calibrated Brier=0.3982 vs raw=0.4121; calibrated ECE=0.0547 vs raw=0.0984. Fit on val_natural only (never test). Calibration improves both metrics -> use calibrated probabilities downstream."
}


## Gate 17: CAMeLBERT-Mix challenger

Official model card verified before running (`CAMeL-Lab/bert-base-arabic-camelbert-mix`, Apache-2.0, pretrained on a MSA+Classical+Dialectal Arabic mix -- the model card makes **no claim** about Arabic-English code-switching support, and this project does not repeat that unsupported claim). Same split/label-contract/max-length/loss-variant/epoch-budget as MARBERT (not re-tuned per model).

In [12]:
card_path = REPORTS_DIR / "challenger_model_card_verification.json"
print(json.loads(card_path.read_text(encoding="utf-8"))["decision"] if card_path.exists() else "not recorded")

ctrain_path = REPORTS_DIR / "challenger_training_manifest.json"
if ctrain_path.exists():
    ctrain = json.loads(ctrain_path.read_text(encoding="utf-8"))
    print("Challenger final_eval_metrics:", json.dumps(ctrain["final_eval_metrics"], indent=2))
else:
    print("Challenger run not present / skipped -- see final modeling report for the reason.")

Proceed with challenger run: license is recorded and permissive (Apache-2.0), model loads via standard AutoModelForSequenceClassification, compute budget is comparable to MARBERT run (same protocol reused per Gate 17), and it answers a genuinely distinct pretraining-corpus question.
Challenger final_eval_metrics: {
  "eval_loss": 0.7870704531669617,
  "eval_macro_f1": 0.6297856393234069,
  "eval_weighted_f1": 0.7243492350341918,
  "eval_negative_f1": 0.6301035953686777,
  "eval_neutral_mixed_f1": 0.43267437523312197,
  "eval_positive_f1": 0.8265789473684211,
  "eval_negative_precision": 0.6018626309662398,
  "eval_negative_recall": 0.6611253196930946,
  "eval_neutral_mixed_precision": 0.3805774278215223,
  "eval_neutral_mixed_recall": 0.5012964563526361,
  "eval_positive_precision": 0.8778647288988262,
  "eval_positive_recall": 0.7809547488811537,
  "eval_runtime": 10.7857,
  "eval_samples_per_second": 552.678,
  "eval_steps_per_second": 2.225,
  "epoch": 3.0
}


## Gates 18/19: HARD / ASTD transfer experiments

In [13]:
print(inv["hard_hotel_reviews"])
print()
print("HARD->LABR transfer (Gate 18): SKIPPED -- HARD is unavailable locally (no data files found, registry status QUARANTINE_LICENSE / files_obtained=false), re-confirmed by fresh filesystem search this session.")
print("ASTD intermediate-stage transfer (Gate 19): see final modeling report for the decision and reasoning.")

{'role': 'UNAVAILABLE, confirmed this session', 'quarantine_data_dir_exists': False, 'any_hard_file_found_under_data': False, 'registry_status': "hard_hotel_reviews\n  canonical_name: HARD\n  portfolio_tier: QUARANTINE\n  status: QUARANTINE_LICENSE\n  files_obtained: false\n  verified_n: null\n  language: Arabic\n  egypt_specific: NOT_PROVEN\n  platform: Booking.com\n  commerce_relevance: MEDIUM\n  task: ratings/sentiment\n  data_license: NOT_FOUND\n  paper_license: SPRINGER_PAPER_RIGHTS\n  code_license: N/A_OR_UNKNOWN\n  source_platform_terms: UNKNOWN_OR_NOT_APPLICABLE\n  commercial_use_status: NOT_READY\n  redistribution_status: DEFERRED_TO_RIGHTS_REVIEW\n  pii_risk: MEDIUM\n  final_role: ARABIC_GENERAL_REVIEW_BENCHMARK\n  primary_source_verified: 'NO'\n"}

HARD->LABR transfer (Gate 18): SKIPPED -- HARD is unavailable locally (no data files found, registry status QUARANTINE_LICENSE / files_obtained=false), re-confirmed by fresh filesystem search this session.
ASTD intermediate-stage 

## Gate 21: Final test evaluation (frozen model, evaluated once)

In [14]:
final_eval = load_json("final_test_evaluation.json")
for split in ["val_natural", "val_balanced", "test_natural", "item_holdout_stress"]:
    m = final_eval[split]
    print(f"{split}: n={m['n']} macro_f1={m['macro_f1']:.4f} weighted_f1={m['weighted_f1']:.4f} "
          f"balanced_acc={m['balanced_accuracy']:.4f} mcc={m['mcc']:.4f} ece={m['ece']:.4f}")
    print("  per_class:", m["per_class"])

val_natural: n=5961 macro_f1=0.6341 weighted_f1=0.7214 balanced_acc=0.6550 mcc=0.4600 ece=0.0551
  per_class: {'negative': {'precision': 0.6315136476426799, 'recall': 0.6508951406649617, 'f1': 0.6410579345088161, 'support': 782}, 'neutral_mixed': {'precision': 0.36806342015855037, 'recall': 0.5617977528089888, 'f1': 0.4447485460143688, 'support': 1157}, 'positive': {'precision': 0.8928887577456477, 'recall': 0.7523620089507708, 'f1': 0.8166239373903657, 'support': 4022}}
val_balanced: n=2346 macro_f1=0.6479 weighted_f1=0.6479 balanced_acc=0.6458 mcc=0.4699 ece=0.1058
  per_class: {'negative': {'precision': 0.7619760479041916, 'recall': 0.6508951406649617, 'f1': 0.7020689655172414, 'support': 782}, 'neutral_mixed': {'precision': 0.5148279952550415, 'recall': 0.5549872122762148, 'f1': 0.5341538461538462, 'support': 782}, 'positive': {'precision': 0.6850299401197605, 'recall': 0.731457800511509, 'f1': 0.7074829931972789, 'support': 782}}
test_natural: n=5962 macro_f1=0.6451 weighted_f1=0.

### Baseline vs. MARBERT direct comparison (identical rows)

In [15]:
comp = load_json("baseline_vs_marbert_comparison.json")
print(json.dumps(comp, indent=2))

{
  "val_natural": {
    "n_rows_compared": 5961,
    "baseline_macro_f1": 0.5147853941945427,
    "marbert_macro_f1": 0.6341434726378502,
    "delta": 0.11935807844330748
  },
  "val_balanced": {
    "n_rows_compared": 2346,
    "baseline_macro_f1": 0.4440761646690789,
    "marbert_macro_f1": 0.6479019349561222,
    "delta": 0.20382577028704335
  },
  "test_natural": {
    "n_rows_compared": 5962,
    "baseline_macro_f1": 0.49637214386753076,
    "marbert_macro_f1": 0.6450811117705005,
    "delta": 0.14870896790296978
  },
  "item_holdout_stress": {
    "n_rows_compared": 439,
    "baseline_macro_f1": 0.4452034298849961,
    "marbert_macro_f1": 0.6590650529778806,
    "delta": 0.21386162309288453
  }
}


## Gate 22: External robustness (ASTD / ArSAS) -- cross-domain stress evidence ONLY

**These are NOT Egyptian e-commerce performance numbers.** Different domain (Twitter, not book reviews or e-commerce), different label semantics (ASTD's OBJ excluded; ArSAS's Mixed folded into Neutral/Mixed). Reported separately and never used to retune the frozen model.

In [16]:
ext_path = REPORTS_DIR / "external_robustness_report.json"
if ext_path.exists():
    ext = json.loads(ext_path.read_text(encoding="utf-8"))
    print("ASTD:", {k: v for k, v in ext["astd"].items() if k in ("macro_f1","n","n_obj_excluded","label_mapping")})
    print("ArSAS:", {k: v for k, v in ext["arsas"].items() if k in ("macro_f1","n","label_mapping")})
    print()
    print(ext["IMPORTANT_CAVEAT"])
else:
    print("External robustness eval not yet run.")

ASTD: {'n': 3315, 'macro_f1': 0.43742045016864967, 'n_obj_excluded': 6691, 'label_mapping': 'POS->Positive, NEG->Negative, NEUTRAL->Neutral/Mixed; OBJ EXCLUDED (not sentiment-equivalent)'}
ArSAS: {'n': 19897, 'macro_f1': 0.4136523150283009, 'label_mapping': 'Negative->Negative, Positive->Positive, Neutral->Neutral/Mixed, Mixed->Neutral/Mixed'}

These are CROSS-DOMAIN ROBUSTNESS STRESS results on Twitter-sourced Arabic sentiment datasets with DIFFERENT label semantics and DIFFERENT domain (social media, not book reviews, not e-commerce) than the primary LABR-trained model. They are NOT a measure of Egyptian e-commerce production readiness and must never be reported as such. Label mapping differences (ASTD's OBJ exclusion, ArSAS's Mixed->Neutral/Mixed fold) mean these numbers are not directly comparable to the primary LABR test metrics either -- they measure a genuinely different question (does the LABR-trained model transfer at all to dialectal/social-media text) and are reported separa

## Gate 23: Statistical significance (paired bootstrap, >=1000 resamples)

In [17]:
sig_path = REPORTS_DIR / "statistical_significance.json"
if sig_path.exists():
    sig = json.loads(sig_path.read_text(encoding="utf-8"))
    print(json.dumps(sig, indent=2))
else:
    print("Significance testing not yet run.")

{
  "marbert_vs_baseline_test_natural": {
    "n_resamples": 1000,
    "n_rows": 5962,
    "point_delta_b_minus_a": 0.14870896790296978,
    "ci_95_lo": 0.13316094300271256,
    "ci_95_hi": 0.16596319287599048,
    "ci_excludes_zero": true,
    "interpretation": "95% CI [0.1332, 0.1660] excludes zero -> the delta is statistically distinguishable from zero."
  },
  "challenger_vs_marbert_test_natural": {
    "skipped": true,
    "reason": "challenger predictions not present (Gate 17 not run or challenger declined)"
  }
}


## Gate 24: Error analysis (post-hoc, no retraining after this)

In [18]:
err_path = REPORTS_DIR / "error_analysis.json"
if err_path.exists():
    err = json.loads(err_path.read_text(encoding="utf-8"))
    print("Total errors:", err["total_errors"], "/", err["total_test_rows"], f"({err['error_rate']*100:.1f}%)")
    print("Sample stratified by true class:", err["sample_stratified_by_true_class"])
    print("Tag frequency:", err["tag_frequency_in_sample"])
else:
    print("Error analysis not yet run.")

Total errors: 1730 / 5962 (29.0%)
Sample stratified by true class: {'Negative': 50, 'Neutral/Mixed': 50, 'Positive': 50}
Tag frequency: {'negation_present': 94, 'neutral_mixed_true_class': 50, 'rating3_text_ambiguity_candidate': 50, 'no_heuristic_pattern_matched': 25, 'long_possibly_truncated': 18, 'extreme_miss_negative_vs_positive': 12, 'letter_repetition': 11, 'very_short': 7}


## Gate 25: Model selection

Selection hierarchy (validation evidence only): validation macro-F1 -> Neutral/Mixed F1 -> robustness -> calibration -> runtime/complexity. See the final modeling report for the explicit selection decision and reasoning.

## Gate 28: Inference functions demo (loads the frozen artifact, single + batch)

In [19]:
from src.nlp.arabic_foundation.inference import ArabicSentimentFoundationModel

final_model_dir = ARTIFACT_DIR / "primary_model" / "final"
if (final_model_dir / "config.json").exists():
    model = ArabicSentimentFoundationModel.load()
    demo_texts = [
        "الكتاب رائع جدا وأنصح به بشدة",
        "كتاب سيء جدا ومضيعة للوقت والمال",
        "الكتاب عادي مش وحش ومش كويس أوي",
    ]
    for r in model.predict_batch(demo_texts):
        print(f"{r.text!r} -> {r.predicted_label_name} (probs={[round(p,3) for p in r.raw_probabilities]})")
else:
    print("Frozen model artifact not present in this environment.")

C:\Users\User2\Desktop\ecommerce_medusa\commerce-pilot-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7768.30it/s]

'الكتاب رائع جدا وأنصح به بشدة' -> Positive (probs=[0.003, 0.026, 0.972])
'كتاب سيء جدا ومضيعة للوقت والمال' -> Negative (probs=[0.99, 0.006, 0.003])
'الكتاب عادي مش وحش ومش كويس أوي' -> Negative (probs=[0.901, 0.094, 0.005])


## Limitations (see model card for the full list)

- **Domain bias**: trained on Goodreads book reviews (LABR), not e-commerce or delivery-service text. Vocabulary, register, and sentiment expression patterns differ from Egyptian e-commerce reviews.
- **Dialectal coverage**: LABR is predominantly MSA with some dialectal Arabic; heavy Egyptian-dialect e-commerce text is not directly represented in training data (ASTD/ArSAS robustness evals are the closest proxy available, and even those are Twitter-domain, not e-commerce).
- **Class imbalance**: LABR's rating distribution skews heavily positive (67% Positive after mapping); Neutral/Mixed is the hardest, lowest-support class even after class-weighted loss.
- **Label ambiguity**: the 3-star -> Neutral/Mixed mapping conflates genuinely mixed sentiment with lukewarm-but-directional opinions -- a known property of star-rating-derived labels.
- **No chronological validation**: LABR has no timestamp field, so temporal drift cannot be assessed.
- **Code-switch / Arabizi**: not a dedicated evaluation axis; LABR contains limited English code-switching and no Arabizi to speak of, so robustness there is unverified.

## Future recommended step: Egyptian e-commerce domain adaptation

This model is a **foundation**, not a deployable Egyptian e-commerce sentiment model. The recommended next step (described only, not started, per this task's scope) is a labeled-data collection + intermediate domain-adaptive fine-tuning stage on genuine Egyptian e-commerce review text before any production use.

## Artifact locations

- Splits: `artifacts/experiments/arabic_foundation/splits/`
- Baseline: `artifacts/experiments/arabic_foundation/baseline/`
- Primary model: `artifacts/experiments/arabic_foundation/primary_model/final/`
- Challenger (if run): `artifacts/experiments/arabic_foundation/challenger/final/`
- All reports/metrics/audits: `reports/generated/arabic_foundation/`
- Model card: `reports/generated/arabic_foundation/ARABIC_FOUNDATION_MODEL_CARD.md`
- Final modeling report: `reports/generated/arabic_foundation/ARABIC_FOUNDATION_FINAL_MODELING_REPORT.md`
- Reproducibility manifest: `reports/generated/arabic_foundation/ARABIC_FOUNDATION_REPRODUCIBILITY_MANIFEST.md`